## PCS956 time series companion A: inspection and basic EDA

This notebook is 'companion A' for the PCS956 time series lectures. It focuses on
how to go from a time-stamped data file to a first inspection of:

- the structure of the time index and sampling pattern
- aspects of data quality that are visible from the data alone
- simple plots of levels and differences
- basic autocorrelation diagnostics
- simple decompositions and persistence baselines

The emphasis here is on well-behaved example series that let us see how
inspection and basic modelling workflows look in a straightforward setting.
In real applications we may encounter much messier data. The intention is
that you start from the templates in this notebook and adapt them carefully
to your own setting, using domain knowledge where needed.

Spectral (frequency-domain) methods and more advanced models will be covered in
later companions.

## Data sources and formats

The datasets in this companion are derived from the R package `astsa`:

> Shumway, R.H. and Stoffer, D.S. `astsa`: Applied Statistical Time Series Analysis  
> GitHub: https://github.com/nickpoison/astsa  
> CRAN: https://cran.r-project.org/package=astsa  

They have been exported to CSV and placed in a local `data/` folder:

- `../data/soi.csv`        - Southern Oscillation Index (monthly, univariate)  
- `../data/ENSO.csv`       - ENSO index (monthly, can be treated as multivariate)  
- `../data/gtemp_both.csv` - global temperature anomalies (annual, univariate)  
- `../data/djia.csv`       - Dow Jones Industrial Average (daily, multivariate)  
- `../data/eqexp.csv`      - seismic traces for earthquake vs explosion (classification-type)  

Each example section below is self-contained: it includes its own imports,
reads one CSV file, and performs basic inspection and EDA. You can copy any
section into your own notebook and adapt the path and column names.

These `astsa` datasets illustrate how workflows look when the time
index behaves in a simple way and the series are well-suited for
basic time-series inspection and modelling.

In the later constructed examples (Sections 7–8), we use simulated series with:

- **missing values** (isolated points and contiguous blocks), and
- **level shifts vs isolated spikes**,

to make it easier to recognise these features when they occur in real data.

The goal in this companion is not to fit sophisticated models, but to:

- develop a systematic habit of inspecting the time index, data quality,
  and basic temporal structure, and
- use simple tools (plots, rolling summaries, ACF/PACF, decompositions,
  and persistence baselines) to distinguish between long-run trends,
  seasonal patterns, level shifts, and more noise-like behaviour.

Later companions will build on this workflow with more advanced models and
spectral (frequency-domain) methods.


### Overview of the inspection workflow

Each template in this companion follows the same basic inspection workflow.
Starting from a time-stamped data file (typically CSV), we move through:

1. **Loading and basic structure**
   - Read the data from file.
   - Ensure that the time index is correctly represented (for example,
     as a `DatetimeIndex` for calendar data, or as an integer index for
     sample sequences).
   - Check the ordering and inferred sampling frequency.

2. **Data-quality checks**
   - Count missing values and check for duplicated time stamps.
   - Inspect simple summary statistics (minimum, maximum, mean, quartiles).
   - Identify obvious glitches or physically implausible values.

3. **Visual inspection of levels and differences**
   - Plot the series in levels to see overall behaviour.
   - Plot first differences (or returns) to see how changes behave.
   - Use rolling mean and rolling standard deviation to highlight changes in
     typical level or volatility over time.

4. **Autocorrelation diagnostics**
   - Plot autocorrelation (ACF) and partial autocorrelation (PACF) of levels.
   - Plot ACF/PACF of differences or returns.
   - Compare how much temporal dependence remains after differencing.

5. **Simple decompositions and baselines (where appropriate)**
   - For regular seasonal data (such as monthly SOI), perform a simple
     additive decomposition into trend, seasonal, and residual components.
   - Inspect residuals and their ACF to see what structure remains after
     removing trend and seasonal patterns.
   - For forecasting-style problems, evaluate a very simple persistence
     baseline (one-step-ahead forecast equal to the last observed value)
     and inspect the baseline residuals and their autocorrelation.

6. **Problem-specific views**
   - For multivariate series (such as ENSO), look at correlations between
     components and focus EDA on one chosen variable.
   - For classification-style data (such as earthquake vs explosion traces),
     compare representative traces from different groups, and consider how
     time-series signals might be turned into features.

In the later constructed examples (Sections 7–8), we use simulated series with:

- **missing values** (isolated points and contiguous blocks), and
- **level shifts vs isolated spikes**,

to make it easier to recognise these features when they occur in real data.

The goal in this companion is not to fit sophisticated models, but to:

- develop a systematic habit of inspecting the time index, data quality,
  and basic temporal structure, and
- use simple tools (plots, rolling summaries, ACF/PACF, decompositions,
  and persistence baselines) to distinguish between long-run trends,
  seasonal patterns, level shifts, and more noise-like behaviour.

Later companions will build on this workflow with more advanced models and
spectral (frequency-domain) methods.


### How this companion relates to the lectures and other companions

This notebook corresponds to **Lecture TS1** (intro and overview) and focuses on:

- temporal indexing and sampling structure,
- data quality, missing values, anomalies, and level shifts,
- visual EDA for levels, differences, trend, seasonality, and autocorrelation,
- simple decompositions and a persistence baseline,
- basic intuition for stationarity and random-walk-like behaviour.

Later companions build on this foundation:

- `PCS956-TS-companion_B` (Lecture TS2):
  classical models and forecasting baselines, ARIMA-type models, and temporal validation on
  levels vs differences.

- `PCS956-TS-companion_C` (Lecture TS3):
  anomaly detection, concept drift, multivariate dependence, and time-aware ML methods.

When working on the mini-project, start from the inspection and EDA patterns in Companion A,
then move to baselines and models in Companion B and ML methods in Companion C as needed.


## 1. Template: univariate monthly series (example: SOI)

This template shows how to:

- load a univariate series from CSV (`soi.csv`)
- inspect structure and basic data quality
- plot levels and differences
- plot ACF/PACF
- perform a simple decomposition to separate trend and periodic structure
- evaluate a persistence baseline

We will later extend this SOI template with basic residual diagnostics
after decomposition and baselines.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_datetime64_any_dtype

from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf

plt.style.use("seaborn-v0_8")

# ---------------------------
# 1.1 Load SOI data
# ---------------------------

soi_path = "../data/soi.csv"   # EDIT if your file is elsewhere

df_soi_raw = pd.read_csv(soi_path)
print("SOI raw head:")
display(df_soi_raw.head())

df_soi = df_soi_raw.copy()

# Ensure datetime index
if "date" in df_soi.columns:
    if not is_datetime64_any_dtype(df_soi["date"]):
        df_soi["date"] = pd.to_datetime(df_soi["date"])
    df_soi = df_soi.set_index("date").sort_index()
else:
    print("No 'date' column found in SOI; using default index.")

# Extract target series
soi = df_soi["soi"].copy()

print("\nSOI info:")
print(soi.to_frame().info())

# ---------------------------
# 1.2 Structural checks
# ---------------------------

n_soi = len(soi)
start_soi = soi.index.min()
end_soi = soi.index.max()

try:
    freq_soi = pd.infer_freq(soi.index)
except Exception:
    freq_soi = None

print(f"\nSOI length: {n_soi}")
print(f"SOI start:  {start_soi}")
print(f"SOI end:    {end_soi}")
print(f"Inferred frequency (SOI): {freq_soi}")

print("\nSOI first 3:")
display(soi.head(3))
print("\nSOI last 3:")
display(soi.tail(3))

# ---------------------------
# 1.3 Data-quality checks
# ---------------------------

print("\nSOI missing values:", soi.isna().sum())
print("\nSOI summary statistics:")
display(soi.describe().to_frame().T)

dup_idx_soi = soi.index.duplicated().sum()
print(f"\nSOI duplicated index values: {dup_idx_soi}")

print("\nSOI range:")
print("  min =", soi.min())
print("  max =", soi.max())

# ---------------------------
# 1.4 Visual EDA: levels, differences, autocorrelation
# ---------------------------

soi_subset = soi   # optionally restrict by date, for example soi["1980":]
soi_diff = soi_subset.diff()

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].plot(soi_subset.index, soi_subset.values, color="tab:blue")
axes[0, 0].set_title("SOI: level")
axes[0, 0].set_xlabel("Time")
axes[0, 0].set_ylabel("SOI")

axes[0, 1].plot(soi_diff.index, soi_diff.values, color="tab:orange")
axes[0, 1].set_title("SOI: first difference")
axes[0, 1].set_xlabel("Time")
axes[0, 1].set_ylabel("Δ SOI")

autocorrelation_plot(soi_subset.dropna(), ax=axes[1, 0])
axes[1, 0].set_title("SOI: autocorrelation (level)")

autocorrelation_plot(soi_diff.dropna(), ax=axes[1, 1])
axes[1, 1].set_title("SOI: autocorrelation (difference)")

plt.tight_layout()
plt.show()

# ---------------------------
# 1.5 Rolling mean and std
# ---------------------------

ROLLING_WINDOW_SOI = 12   # 12 months

soi_roll_mean = soi_subset.rolling(ROLLING_WINDOW_SOI).mean()
soi_roll_std = soi_subset.rolling(ROLLING_WINDOW_SOI).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(
    soi_subset.index,
    soi_subset.values,
    label="SOI",
    color="tab:blue",
)
axes[0].plot(
    soi_roll_mean.index,
    soi_roll_mean.values,
    label=f"Rolling mean (window={ROLLING_WINDOW_SOI})",
    color="tab:orange",
)
axes[0].set_title("SOI: level and rolling mean")
axes[0].set_ylabel("SOI")
axes[0].legend(fontsize=8)

axes[1].plot(
    soi_roll_std.index,
    soi_roll_std.values,
    label=f"Rolling std (window={ROLLING_WINDOW_SOI})",
    color="tab:green",
)
axes[1].set_title("SOI: rolling standard deviation")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Std. dev.")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Short note: rolling windows for mean and standard deviation

In several examples we use **rolling windows** to compute running summaries such as:

- rolling mean (average over a recent window),
- rolling standard deviation (a simple measure of local variability).

For a window of size $w$ (for example $w = 12$ months), the rolling mean at time $t$ is:

$$
\bar{X}_t^{(w)} = \frac{1}{w} \sum_{j=0}^{w-1} X_{t-j},
$$

and the rolling standard deviation is the usual sample standard deviation computed over the same
window. These summaries are helpful for:

- visualising **slow changes in typical level** (trend or regime changes),
- seeing **changes in variability** (for example calm vs turbulent periods).

Rolling windows have some important limitations:

- the choice of window size $w$ is **arbitrary and context-dependent**:
  - too small: very noisy behaviour;
  - too large: genuine changes can be smoothed away.
- they are primarily **exploratory tools**, not formal tests; they give hints about changes, but do
  not prove that a process has become stationary or non-stationary.
- near the start of the series, rolling statistics are based on fewer points, so edge behaviour
  should be interpreted with caution.

In this companion rolling means and standard deviations are used mainly as **visual diagnostics** to
highlight possible changes in level or volatility, not as modelling assumptions.


In [ ]:

# ---------------------------
# 1.6 ACF and PACF
# ---------------------------

max_lag_soi = 40

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

plot_acf(soi_subset.dropna(), lags=max_lag_soi, ax=axes[0, 0])
axes[0, 0].set_title("SOI: ACF (level)")

plot_pacf(soi_subset.dropna(), lags=max_lag_soi, ax=axes[0, 1], method="ywm")
axes[0, 1].set_title("SOI: PACF (level)")

plot_acf(soi_diff.dropna(), lags=max_lag_soi, ax=axes[1, 0])
axes[1, 0].set_title("SOI: ACF (first difference)")

plot_pacf(soi_diff.dropna(), lags=max_lag_soi, ax=axes[1, 1], method="ywm")
axes[1, 1].set_title("SOI: PACF (first difference)")

plt.tight_layout()
plt.show()

### Autocorrelation with anomalies and missing values

In these curated examples the series themselves are **clean**: there are no genuine missing values
in the original data files, and no obvious anomalies. Any `NaN` values we encounter come from
simple transformations such as:

- first differences (`series.diff()`), which introduce a `NaN` at the start,
- decompositions, which can produce `NaN` at the ends of trend or residual components.

Before computing autocorrelations or partial autocorrelations we therefore use:

- `series.dropna()` to remove those transformation-induced `NaN` values, and
- pass only the finite-valued series to functions such as `plot_acf`, `plot_pacf`, and
  `autocorrelation_plot`.

In real applications we may face two additional issues:

- **Anomalies (spikes, glitches)**:
  - A few large spikes can distort sample autocorrelations, especially at short lags.
  - Before treating ACF/PACF as describing “typical behaviour”, we should inspect the series and
    decide whether obvious anomalies should be removed, adjusted, or explicitly discussed as part
    of the story (for example, interventions or rare events).

- **Genuine missing values (`NaN`) in the raw data**:
  - Standard time-series tools (autocorrelations, spectral estimates, many model fits) do not work
    with `NaN` values; they require finite observations.
  - Simply dropping all missing values (`dropna()`) can be acceptable when there are **very few
    isolated gaps in a long series**, but it may change the effective sampling pattern and reduce
    reliability when gaps are more substantial.

In this companion we:

- compute ACF/PACF only on finite-valued series (after removing the transformation-induced `NaN`s),
- treat serious missingness and anomalies as **features to be described** (see Sections 7–8),

rather than silently ignoring them. In your own mini-project, you should explicitly describe how
missing values and anomalies are handled before computing autocorrelations or spectra, and avoid
using “ignore NA” defaults without thinking about their effect on the analysis.



In [ ]:

# ---------------------------
# 1.6b Optional: stationarity tests (ADF) for SOI
# ---------------------------

from statsmodels.tsa.stattools import adfuller

def adf_summary(series, label):
    """Run ADF test and print a compact summary."""
    result = adfuller(series.dropna(), autolag="AIC")
    test_stat, p_value, used_lag, n_obs, crit_values, _ = result

    print(f"\nADF test ({label}):")
    print(f"  Test statistic: {test_stat:.3f}")
    print(f"  p-value:        {p_value:.3f}")
    print(f"  Used lags:      {used_lag}")
    print(f"  Number of obs.: {n_obs}")
    print("  Critical values:")
    for k, v in crit_values.items():
        print(f"    {k}: {v:.3f}")

    if p_value < 0.05:
        print("  Interpretation: reject unit-root null at 5% level "
              "(evidence for stationarity).")
    else:
        print("  Interpretation: cannot reject unit-root null at 5% level "
              "(series may be non-stationary).")

# ADF on SOI level and first difference
adf_summary(soi_subset, "SOI level")
adf_summary(soi_diff, "SOI first difference")

### Optional note: formal stationarity tests

The Augmented Dickey–Fuller (ADF) test is one way to check for a unit root in
a time series. In its simplest form, it tests:

$$
H_0: \text{series has a unit root (is non-stationary)} \\
H_1: \text{series is stationary (no unit root).}
$$

In this notebook, we use the ADF test as a **supporting tool**:

- we already look at levels, differences, and ACF/PACF plots,
- the ADF test can be used to see whether these visual impressions are
  consistent with a formal unit-root test.

It is important to remember that:

- the ADF test has assumptions (for example, about the form of the trend and
  the error structure),
- failing to reject $H_0$ does not prove that a series is “non-stationary” in
  every sense,
- passing the test does not guarantee that the series behaves like a simple,
  stationary model.

In practice, tests such as ADF are used alongside visual diagnostics and domain
knowledge, not as a substitute for them.

In Lecture TS1 (Section 6.1) we discuss tests such as ADF as heuristics for
unit-root behaviour. Here you can see how they line up with the visual
diagnostics in this companion, treating them as guides rather than oracles.


In [ ]:

# ---------------------------
# 1.7 Simple seasonal decomposition (monthly SOI)
# ---------------------------

try:
    soi_decomp = seasonal_decompose(
        soi_subset.dropna(),
        model="additive",
        period=12,
    )

    # Components: observed, trend, seasonal, residual
    fig = soi_decomp.plot()
    fig.set_size_inches(10, 8)
    fig.suptitle("SOI: seasonal decomposition", y=1.02)
    plt.tight_layout()
    plt.show()

    # ---------------------------
    # 1.7a Residual diagnostics after decomposition
    # ---------------------------

    soi_resid = soi_decomp.resid.dropna()

    fig, axes = plt.subplots(2, 1, figsize=(10, 6))

    # Residuals over time
    axes[0].plot(
        soi_resid.index,
        soi_resid.values,
        color="tab:red",
    )
    axes[0].axhline(0.0, color="black", linewidth=0.8)
    axes[0].set_title("SOI: residuals after trend + seasonal decomposition")
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel("Residual")

    # ACF of residuals
    plot_acf(soi_resid, lags=max_lag_soi, ax=axes[1])
    axes[1].set_title("SOI: ACF of decomposition residuals")
    axes[1].set_xlabel("Lag")

    plt.tight_layout()
    plt.show()

except ValueError as e:
    print("SOI decomposition failed:", e)

# ---------------------------
# 1.8 Persistence baseline (one-step)
# ---------------------------

soi_shift = soi_subset.shift(1)
soi_resid_baseline = soi_subset - soi_shift

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(
    soi_subset.index,
    soi_subset.values,
    label="Actual",
    color="tab:blue",
)
axes[0].plot(
    soi_shift.index,
    soi_shift.values,
    label="Persistence",
    color="tab:orange",
)
axes[0].set_title("SOI: persistence baseline (one-step ahead)")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("SOI")
axes[0].legend(fontsize=8)

axes[1].plot(
    soi_resid_baseline.index,
    soi_resid_baseline.values,
    color="tab:red",
)
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set_title("SOI: baseline residuals (actual - persistence)")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()

print("\nSOI persistence baseline:")
print("  MAE:", soi_resid_baseline.abs().mean())
print("  RMSE:", np.sqrt((soi_resid_baseline**2).mean()))

# ---------------------------
# 1.8a ACF of persistence-baseline residuals
# ---------------------------

fig, ax = plt.subplots(figsize=(8, 4))

plot_acf(soi_resid_baseline.dropna(), lags=max_lag_soi, ax=ax)
ax.set_title("SOI: ACF of persistence-baseline residuals")
ax.set_xlabel("Lag")

plt.tight_layout()
plt.show()

## 2. Template: multivariate monthly series (example: ENSO)

This template shows:

- loading `ENSO.csv` as a multivariate time series
- basic inspection and correlation
- EDA for one chosen variable

Later extensions could include:

- a quick “spaghetti” plot of all numeric ENSO variables
- a basic scatter plot to illustrate dependence between two components


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_datetime64_any_dtype

from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.style.use("seaborn-v0_8")

enso_path = "../data/ENSO.csv"   # EDIT if needed

df_enso_raw = pd.read_csv(enso_path)
print("ENSO raw head:")
display(df_enso_raw.head())

df_enso = df_enso_raw.copy()

if "date" in df_enso.columns:
    if not is_datetime64_any_dtype(df_enso["date"]):
        df_enso["date"] = pd.to_datetime(df_enso["date"])
    df_enso = df_enso.set_index("date").sort_index()

print("\nENSO info:")
print(df_enso.info())

# Correlation (numeric columns only)
print("\nENSO correlation matrix (numeric columns):")
display(df_enso.select_dtypes(include=[np.number]).corr())

### Multivariate structure and cross-correlations (conceptual note)

For multivariate series, we care about **both**:

- how each component behaves over time (its own trend, seasonality, autocorrelation), and
- how components relate to each other (for example, whether changes in one tend to precede or
  follow changes in another).

The simple correlation matrix above shows **static correlations** between components. In later
materials (TS2/TS3 and their companions) we will also look at:

- **cross-correlation functions**, which describe how one series is related to lagged versions of
  another,
- and Granger-style predictive “causality” tools (still about prediction, not strong causal claims).

In this companion, the multivariate example is kept light:

- use the correlation matrix to see which variables move together on average,
- and then focus EDA on one chosen variable (`ENSO` or a similar index).


In [ ]:
# Choose one main variable for EDA (students can change this)
if "ENSO" in df_enso.columns:
    enso_main = df_enso["ENSO"].copy()
else:
    # Fallback: first numeric column
    num_cols_enso = df_enso.select_dtypes(include=[np.number]).columns
    enso_main = df_enso[num_cols_enso[0]].copy()
    print(f"\nUsing {num_cols_enso[0]} as main ENSO variable.")

enso_diff = enso_main.diff()

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].plot(enso_main.index, enso_main.values, color="tab:blue")
axes[0, 0].set_title("ENSO: level")
axes[0, 0].set_xlabel("Time")
axes[0, 0].set_ylabel("ENSO")

axes[0, 1].plot(enso_diff.index, enso_diff.values, color="tab:orange")
axes[0, 1].set_title("ENSO: first difference")
axes[0, 1].set_xlabel("Time")
axes[0, 1].set_ylabel("Δ ENSO")

autocorrelation_plot(enso_main.dropna(), ax=axes[1, 0])
axes[1, 0].set_title("ENSO: autocorrelation (level)")

autocorrelation_plot(enso_diff.dropna(), ax=axes[1, 1])
axes[1, 1].set_title("ENSO: autocorrelation (difference)")

plt.tight_layout()
plt.show()

### Plotting multivariate series: when simple plots are enough (and when they are not)

In this small ENSO example, plotting **one main variable** over time (with its differences and
autocorrelation) is usually enough to illustrate basic time-series behaviour.

For multivariate series more generally, there are several common plotting strategies:

- **Spaghetti plots**: overlay several components in one plot.
  - Helpful when there are only a few series with compatible units and similar scales.
  - Can become hard to read if there are many dimensions, very long series, or very different
    magnitudes.

- **Small multiples**: separate plots for each component (or for a few selected components),
  arranged side by side or in a grid.
  - Clearer when units or scales differ, or when we want to focus on subsets of variables.
  - Still manageable only for modest numbers of series.

- **Interactive tools or dimension reduction**:
  - When there are many variables, static plots quickly become cluttered.
  - In such cases, interactive plotting, brushing/filtering tools, or dimension-reduction methods
    (for example PCA on features, embeddings) may be more appropriate for exploring structure.

In this companion multivariate plotting is deliberately kept simple, focusing on one main
variable plus a correlation matrix. For high-dimensional series, we are expected to think about:

- which variables are most relevant to our question,
- whether they can reasonably share a plot and axis,
- and whether we need more advanced tools to explore multivariate dependence.


## 3. Template: annual series (example: global temperature)

This template focuses on an annual univariate series with a clear long-term trend.

Here we:

- overlay a simple linear trend fit to highlight non-stationarity in levels,
- add residual checks for a trend-only model (including an ACF of residuals).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_datetime64_any_dtype

from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.style.use("seaborn-v0_8")

gtemp_path = "../data/gtemp_both.csv"

df_gtemp_raw = pd.read_csv(gtemp_path)
print("gtemp_both raw head:")
display(df_gtemp_raw.head())

df_gtemp = df_gtemp_raw.copy()

if "date" in df_gtemp.columns:
    if not is_datetime64_any_dtype(df_gtemp["date"]):
        df_gtemp["date"] = pd.to_datetime(df_gtemp["date"])
    df_gtemp = df_gtemp.set_index("date").sort_index()

gtemp = df_gtemp["gtemp_both"].copy()

print("\ngtemp_both info:")
print(gtemp.to_frame().info())

gtemp_diff = gtemp.diff()

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].plot(gtemp.index, gtemp.values, color="tab:blue")
axes[0, 0].set_title("Global temperature: level")
axes[0, 0].set_xlabel("Year")
axes[0, 0].set_ylabel("Anomaly")

axes[0, 1].plot(gtemp_diff.index, gtemp_diff.values, color="tab:orange")
axes[0, 1].set_title("Global temperature: first difference")
axes[0, 1].set_xlabel("Year")
axes[0, 1].set_ylabel("Δ anomaly")

autocorrelation_plot(gtemp.dropna(), ax=axes[1, 0])
axes[1, 0].set_title("Global temperature: autocorrelation (level)")

autocorrelation_plot(gtemp_diff.dropna(), ax=axes[1, 1])
axes[1, 1].set_title("Global temperature: autocorrelation (difference)")

plt.tight_layout()
plt.show()

# ---------------------------
# 3.1 Simple linear trend fit and residual checks
# ---------------------------

# Prepare a simple numeric time index for regression: 0, 1, 2, ...
time_index = np.arange(len(gtemp))
X = np.vstack([np.ones(len(time_index)), time_index]).T  # intercept + slope

# Ordinary least squares: gtemp ≈ β0 + β1 * t
beta_hat = np.linalg.lstsq(X, gtemp.values, rcond=None)[0]
beta0, beta1 = beta_hat

# Fitted linear trend as a time series aligned with gtemp
gtemp_trend = beta0 + beta1 * time_index
gtemp_trend = pd.Series(gtemp_trend, index=gtemp.index, name="linear_trend")

# Residuals from the trend-only model
gtemp_resid_trend = gtemp - gtemp_trend

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Observed vs linear trend
axes[0].plot(gtemp.index, gtemp.values, color="tab:blue", label="Observed")
axes[0].plot(gtemp_trend.index, gtemp_trend.values, color="tab:red", label="Linear trend")
axes[0].set_title("Global temperature: observed series and fitted linear trend")
axes[0].set_ylabel("Anomaly")
axes[0].legend(fontsize=8)

# Residuals over time
axes[1].plot(gtemp_resid_trend.index, gtemp_resid_trend.values, color="tab:gray")
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set_title("Residuals from linear trend model")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()

# ACF of residuals to see remaining structure after removing the trend
fig, ax = plt.subplots(figsize=(8, 4))
plot_acf(gtemp_resid_trend.dropna(), lags=40, ax=ax)
ax.set_title("Global temperature: ACF of linear-trend residuals")
ax.set_xlabel("Lag")
plt.tight_layout()
plt.show()

## 4. Template: daily financial series (example: DJIA Close)

This template uses a daily financial series, where levels often look like a
random walk and differences or returns are closer to noise.

We see:

- levels and first differences,
- ACF on both,
- log-returns and their autocorrelation (often near zero at most lags, in line with the
  random-walk discussion in the lecture),
- a rolling volatility estimate to visualise calm vs turbulent periods.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_datetime64_any_dtype

from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.style.use("seaborn-v0_8")

djia_path = "../data/djia.csv"

df_djia_raw = pd.read_csv(djia_path)
print("DJIA raw head:")
display(df_djia_raw.head())

df_djia = df_djia_raw.copy()

if "date" in df_djia.columns:
    if not is_datetime64_any_dtype(df_djia["date"]):
        df_djia["date"] = pd.to_datetime(df_djia["date"])
    df_djia = df_djia.set_index("date").sort_index()

# Use closing price as main series
djia_close = df_djia["Close"].copy()

print("\nDJIA Close info:")
print(djia_close.to_frame().info())

djia_diff = djia_close.diff()

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].plot(djia_close.index, djia_close.values, color="tab:blue")
axes[0, 0].set_title("DJIA Close: level")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Close")

axes[0, 1].plot(djia_diff.index, djia_diff.values, color="tab:orange")
axes[0, 1].set_title("DJIA Close: first difference (approx. return)")
axes[0, 1].set_xlabel("Date")
axes[0, 1].set_ylabel("Δ Close")

autocorrelation_plot(djia_close.dropna(), ax=axes[1, 0])
axes[1, 0].set_title("DJIA Close: autocorrelation (level)")

autocorrelation_plot(djia_diff.dropna(), ax=axes[1, 1])
axes[1, 1].set_title("DJIA Close: autocorrelation (difference)")

plt.tight_layout()
plt.show()

# ---------------------------
# 4.1 Log-returns and their autocorrelation
# ---------------------------

# Work with strictly positive prices to avoid issues when taking logs
djia_close_pos = djia_close[djia_close > 0]
djia_log = np.log(djia_close_pos)

log_returns = djia_log.diff()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Log-returns over time
axes[0].plot(log_returns.index, log_returns.values, color="tab:green")
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].set_title("DJIA Close: log-returns")
axes[0].set_ylabel("Log-return")

# ACF of log-returns
plot_acf(log_returns.dropna(), lags=40, ax=axes[1])
axes[1].set_title("DJIA Close: ACF of log-returns")
axes[1].set_xlabel("Lag")

plt.tight_layout()
plt.show()

# ---------------------------
# 4.2 Rolling volatility of log-returns
# ---------------------------

ROLLING_WINDOW_DJIA = 21  # approx. one trading month

rolling_vol = log_returns.rolling(ROLLING_WINDOW_DJIA).std()

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(rolling_vol.index, rolling_vol.values, color="tab:purple")
ax.set_title(
    f"DJIA Close: rolling volatility of log-returns "
    f"(window={ROLLING_WINDOW_DJIA})"
)
ax.set_xlabel("Date")
ax.set_ylabel("Rolling std. of log-returns")

plt.tight_layout()
plt.show()

### Rolling volatility as a visual diagnostic

Here we use a rolling standard deviation of log-returns (over about one trading month) to highlight
periods of higher or lower variability. This follows the same idea as the rolling mean/std
in Section 1.5: a **moving summary** over a chosen window, used for visual diagnostics rather than
formal testing.


### Connection to TS1: random-walk-like behaviour

The DJIA Close series illustrates the TS1 discussion of random walks:
- levels look like a smooth, trending series with apparent structure,
- differences and log-returns are much closer to noise,
- ACF of returns is often near zero at most lags.

This is why persistence baselines and modelling on differences/returns are central in
financial time-series work.


## 5. Template: classification-style data (example: eqexp)

This template:

- loads `eqexp.csv` with sample index `t`
- selects one earthquake channel (for example EQ1) and one explosion channel (EX1)
- plots them for visual comparison

No classification model is fitted here; this is purely EDA.

Later we may add:

- overlay of multiple earthquake traces and multiple explosion traces
- simple features (for example, energy, maximum amplitude) for two example traces,
  to illustrate how time-series signals can be turned into features for
  classification


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

eqexp_path = "../data/eqexp.csv"

df_eqexp_raw = pd.read_csv(eqexp_path)
print("eqexp raw head:")
display(df_eqexp_raw.head())

df_eqexp = df_eqexp_raw.copy()

# Use integer index t as given
if "t" in df_eqexp.columns:
    df_eqexp = df_eqexp.set_index("t").sort_index()
else:
    print("No 't' column in eqexp; using default index.")

print("\neqexp info:")
print(df_eqexp.info())

# Pick one earthquake and one explosion channel
eq_col = "EQ1"
ex_col = "EX1"

if eq_col in df_eqexp.columns and ex_col in df_eqexp.columns:
    eq_series = df_eqexp[eq_col].copy()
    ex_series = df_eqexp[ex_col].copy()

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axes[0].plot(eq_series.index, eq_series.values, color="tab:blue")
    axes[0].set_title(f"{eq_col}: example earthquake trace")
    axes[0].set_ylabel("Amplitude")

    axes[1].plot(ex_series.index, ex_series.values, color="tab:orange")
    axes[1].set_title(f"{ex_col}: example explosion trace")
    axes[1].set_xlabel("Sample index t")
    axes[1].set_ylabel("Amplitude")

    plt.tight_layout()
    plt.show()
else:
    print(f"Columns {eq_col} and/or {ex_col} not found in eqexp.csv.")

## 7. Constructed example with missing values

In many real data sets, some values are missing. Once you see that the count
of missing values is positive, it is useful to look more closely at when and
how these gaps occur. One way to do this is to plot the series and highlight
intervals where values are not observed (for example, using semi-transparent
bands), so that patterns in the missing values become visible.

In this section we construct a simple example with:

- isolated missing values,
- and one or two contiguous blocks of missing values,

and illustrate:

- how these gaps appear in a basic time series plot when intervals with
  missing data are visually marked,
- how to recognise whether missingness tends to occur as isolated points,
  contiguous blocks, or seems to follow some other pattern.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

# ---------------------------
# 7.1 Simulate a simple AR(1) series
# ---------------------------

np.random.seed(123)

n_months = 240  # 20 years
dates = pd.date_range(start="1980-01-01", periods=n_months, freq="MS")

phi = 0.6
sigma = 1.0

x = np.zeros(n_months)
noise = np.random.normal(loc=0.0, scale=sigma, size=n_months)
for t in range(1, n_months):
    x[t] = phi * x[t - 1] + noise[t]

series_clean = pd.Series(x, index=dates, name="sim_ar1_example")

# ---------------------------
# 7.2 Introduce missing values
# ---------------------------

series_missing = series_clean.copy()

# Isolated missing points
isolated_indices = [dates[20], dates[75], dates[150]]
for idx in isolated_indices:
    series_missing.loc[idx] = np.nan

# Contiguous blocks of missing values
block1_start, block1_end = dates[60], dates[65]   # 6 months
block2_start, block2_end = dates[170], dates[174] # 5 months

series_missing.loc[block1_start:block1_end] = np.nan
series_missing.loc[block2_start:block2_end] = np.nan

print("Constructed missingness example:")
print("  Total length:", len(series_missing))
print("  Missing values:", series_missing.isna().sum())

# ---------------------------
# 7.3 Identify contiguous missing intervals for shading
# ---------------------------

is_missing = series_missing.isna()
missing_diff = is_missing.astype(int).diff().fillna(0)

# Start indices where we switch from not-missing to missing
block_starts = series_missing.index[(missing_diff == 1)].tolist()
# End indices where we switch from missing to not-missing
block_ends = series_missing.index[(missing_diff == -1)].tolist()

# If series ends with missing values, close the last block at the end
if is_missing.iloc[-1] and (len(block_ends) < len(block_starts)):
    block_ends.append(series_missing.index[-1])

missing_blocks = list(zip(block_starts, block_ends))

print("\nMissing blocks detected:")
for start, end in missing_blocks:
    print(f"  {start.date()} to {end.date()}")

# ---------------------------
# 7.4 Plot with highlighted missing intervals
# ---------------------------

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(
    series_missing.index,
    series_missing.values,
    marker="o",
    linestyle="-",
    color="tab:blue",
    label="Observed (with missing values)",
)

for start, end in missing_blocks:
    ax.axvspan(
        start,
        end,
        color="gray",
        alpha=0.2,
        label="Missing interval" if start == missing_blocks[0][0] else None,
    )

ax.set_title("Simulated AR(1) series with isolated and block missing values")
ax.set_xlabel("Time")
ax.set_ylabel("Value")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------
# 7.5 Comment placeholder
# ---------------------------

# At this point, students are expected to look at the plot and describe:
# - where isolated missing points occur
# - where contiguous blocks of missingness occur
# - whether missingness appears random or shows any visible pattern.

## 8. Simulated example with a level shift

Changes in the typical level of a series over time (for example, before and
after a policy change or intervention) are easier to see first in a simple,
controlled example. A simulated series with a known level shift can clarify
how such changes appear in basic plots and rolling summaries.

In this section we construct a simple simulated series with:

- a constant mean in an initial period,
- a different mean in a later period (a level shift),

and illustrate:

- how the level change appears in a time series plot,
- how a rolling mean can help to visualise changes in typical level,
- how this differs from short-lived spikes or outliers.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

# ---------------------------
# 8.1 Simulate a series with a level shift
# ---------------------------

np.random.seed(456)

n_months = 360  # 30 years
dates = pd.date_range(start="1980-01-01", periods=n_months, freq="MS")

change_point = 180  # after 15 years
m1 = 0.0
m2 = 2.0
sigma = 1.0

noise = np.random.normal(loc=0.0, scale=sigma, size=n_months)
series_level_shift = np.empty(n_months)

series_level_shift[:change_point] = m1 + noise[:change_point]
series_level_shift[change_point:] = m2 + noise[change_point:]

series_level_shift = pd.Series(series_level_shift, index=dates, name="sim_level_shift")

print("Simulated level-shift example:")
print("  Total length:", len(series_level_shift))
print("  Change-point at:", series_level_shift.index[change_point])

# ---------------------------
# 8.2 Rolling mean to highlight level change
# ---------------------------

ROLLING_WINDOW = 12  # 12 months
rolling_mean = series_level_shift.rolling(ROLLING_WINDOW).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Level and change-point
axes[0].plot(
    series_level_shift.index,
    series_level_shift.values,
    color="tab:blue",
    label="Simulated series",
)
axes[0].axvline(
    series_level_shift.index[change_point],
    color="red",
    linestyle="--",
    label="Change-point",
)
axes[0].set_title("Simulated series with a level shift")
axes[0].set_ylabel("Value")
axes[0].legend(fontsize=8)

# Rolling mean
axes[1].plot(
    rolling_mean.index,
    rolling_mean.values,
    color="tab:orange",
    label=f"Rolling mean (window={ROLLING_WINDOW})",
)
axes[1].axvline(
    series_level_shift.index[change_point],
    color="red",
    linestyle="--",
    label="Change-point",
)
axes[1].set_title("Rolling mean highlighting change in typical level")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Rolling mean")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Rolling mean as a simple local level estimate

The 12-month rolling mean here is just the average over the most recent year:

$$
\bar{X}_t^{(12)} = \frac{1}{12} \sum_{j=0}^{11} X_{t-j}.
$$

It provides a **local estimate of typical level**, which makes the level shift easier to see.
As in other sections, this is an exploratory tool: it helps visualise changes, but does not replace
more formal change-point or regime-shift methods.


In [ ]:

# ---------------------------
# 8.3 Optional comparison: spikes without level shift
# ---------------------------

series_spikes = pd.Series(
    m1 + noise,  # no sustained level change
    index=dates,
    name="sim_spikes_only",
)

# Add a few isolated spikes
spike_indices = [dates[60], dates[150], dates[270]]
for idx in spike_indices:
    series_spikes.loc[idx] += 6.0  # large positive spike

rolling_mean_spikes = series_spikes.rolling(ROLLING_WINDOW).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(
    series_spikes.index,
    series_spikes.values,
    color="tab:blue",
    label="Series with isolated spikes",
)
axes[0].set_title("Series with isolated spikes but no sustained level shift")
axes[0].set_ylabel("Value")
axes[0].legend(fontsize=8)

axes[1].plot(
    rolling_mean_spikes.index,
    rolling_mean_spikes.values,
    color="tab:orange",
    label=f"Rolling mean (window={ROLLING_WINDOW})",
)
axes[1].set_title("Rolling mean remains near constant level despite spikes")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Rolling mean")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------
# 8.4 Comment placeholder
# ---------------------------

# Exercise (for students): compare
# - the series with a genuine level shift (Section 8.1–8.2), where the rolling
#   mean changes clearly after the change-point
# - the series with only isolated spikes (Section 8.3), where the rolling mean
#   is much more stable.
#
# This helps to distinguish short-lived spikes from sustained changes in level.

## 9. Mini-project: how to use these templates

For the mini-project, you can:

- start from one of the templates above (univariate, multivariate, or eqexp),
- adapt the `..._path` and column names to your own CSV file(s),
- repeat the basic steps:
  - structural checks and basic data quality,
  - plots of levels and differences,
  - rolling statistics,
  - ACF/PACF,
  - simple decomposition where appropriate,
  - persistence baseline (for forecasting-type problems),
  - visual comparison of series from different groups (for classification-type problems).

In your own data, you may see features that are not present in these
curated examples, such as:

- irregular or changing sampling intervals,
- missing periods (gaps) or blocks of missing values,
- obvious level shifts or regime changes,
- sensor glitches or physically impossible values.

The intended workflow is:

1. Detect and document such features using simple diagnostic tools
   (missingness indicators, gap plots, rolling mean/variance, and similar).
2. Discuss serious features with someone who understands the domain and how
   the data were collected (for example, an experimentalist, clinician, or engineer).
3. Decide whether:
   - to restrict attention to a clean subset of the data,
   - to treat the detection of these features as an important part of your
     mini-project story,
   - or to make conservative modelling choices that explicitly acknowledge
     the limitations.

This companion focuses on inspecting and describing time series, not on
developing full imputation schemes or specialised models for structural
breaks and regime changes. When there is substantial missingness or clear
changes in behaviour, sensible modelling choices depend on the context and
on domain knowledge.

For the purposes of this mini-project, it is often preferable to:

- document clearly what kinds of features you see (gaps, level shifts, obvious
  glitches), and
- be explicit about how these limitations affect the questions you can and
  cannot answer,

rather than quietly applying ad-hoc transformations and then proceeding as if
the series were well-behaved and stationary when it is not.

Change-point and regime-shift models, when they are used, are tools for
describing and detecting structural changes. They do not turn an inherently
unstable or evolving process into a simple stationary one.

Later companions will build on this foundation with more advanced models
and, separately, spectral (frequency-domain) methods.